In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False


# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import make_scorer
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer, f1_score

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv('상위6개컬럼모음.csv')
test = pd.read_csv('상위6개컬럼모음.test.csv')

In [3]:
X_train=train.drop(['Segment'],axis=1)
y=train['Segment']
X_test = test

In [4]:
# 문자열 클래스일 경우 숫자로 변환
le = LabelEncoder()
y = le.fit_transform(y)

In [5]:
# 데이터 분리하기
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
print(X_train.shape, X_val.shape, y_val.shape, y_train.shape)

(1920000, 5) (480000, 5) (480000,) (1920000,)


In [6]:
#학습
model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='gpu_hist'
)

model.fit(X_train, y_train)
y_val_train = model.predict(X_val)

In [7]:
# 교차검증
f1_micro = make_scorer(f1_score, average='micro')
scores = cross_val_score(model, X_train, y_train, cv=5, scoring=f1_micro)

In [9]:
print("교차검증 평균 F1:", scores.mean())

교차검증 평균 F1: 0.8351979166666667


In [14]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300]
}

grid = GridSearchCV(
    estimator=XGBClassifier(objective='multi:softmax', num_class=5, use_label_encoder=False, eval_metric='mlogloss',
                            tree_method="hist", device="cuda"),
    param_grid=param_grid,
    scoring='f1_micro',
    cv=3
)

grid.fit(X_train, y_train)

print("최적 F1:", grid.best_score_)
print("최적 파라미터:", grid.best_params_)

[11:51:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[11:51:19] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[11:51:36] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[11:51:51] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[11:52:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[11:53:26] WARNING: 

In [19]:
# 튜닝 결과 반영하여 최종 모델 정의
final_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    learning_rate=0.1,
    max_depth=7,
    n_estimators=100,
    use_label_encoder=False
)

In [21]:
# 전체 훈련 데이터로 학습
final_model.fit(X_train, y_train)

# 검증 데이터 성능 평가
y_val_pred = final_model.predict(X_val)
print("최종 F1 (micro):", f1_score(y_val, y_val_pred, average='micro'))
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

최종 F1 (micro): 0.83518125
              precision    recall  f1-score   support

           A       1.00      0.00      0.01       217
           B       0.00      0.00      0.00        27
           C       0.56      0.34      0.42     25550
           D       0.52      0.30      0.38     70177
           E       0.87      0.97      0.92    384029

    accuracy                           0.84    480000
   macro avg       0.59      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000

